<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-07-fine-tuning/lesson-7.3-fine-tuning-with-vertex-ai/notebooks/exercises-7.3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 7.3 — Fine-Tuning with Vertex AI
Netsetos GenAI Course v2.0 | Module 7

Google Cloud Vertex AI managed tuning for Gemini models


In [ ]:
# pip install google-genai google-cloud-storage -q
print('Lesson 7.3: Fine-Tuning with Vertex AI')


## Ex 1: Vertex AI JSONL Data Format


In [ ]:
import json

def create_vertex_example(system, user, model_response):
    example = {
        'contents': [
            {'role': 'user', 'parts': [{'text': user}]},
            {'role': 'model', 'parts': [{'text': model_response}]}
        ]
    }
    if system:
        example['systemInstruction'] = {
            'role': 'system', 'parts': [{'text': system}]
        }
    return example

ex = create_vertex_example(
    'You are a helpful customer support agent.',
    'How do I reset my password?',
    'Go to Settings > Security > Reset Password. You\'ll receive an email within 2 minutes.'
)
print(json.dumps(ex, indent=2))


## Ex 2: Cost Estimator


In [ ]:
def estimate_cost(num_examples, avg_tokens, epochs, rate_per_mtok=3.0):
    total_tokens = num_examples * avg_tokens * epochs
    cost = total_tokens * rate_per_mtok / 1e6
    return cost

print('Vertex AI Fine-Tuning Cost Estimates:')
print()
for name, examples, tokens, epochs in [
    ('Small experiment', 100, 500, 4),
    ('Medium production', 1000, 500, 5),
    ('Large production', 10000, 500, 3),
]:
    flash = estimate_cost(examples, tokens, epochs, 3.0)
    pro = estimate_cost(examples, tokens, epochs, 8.0)
    print(f'  {name:20s}: Flash ~${flash:.2f} | Pro ~${pro:.2f}')
print()
print('Inference: SAME price as base model (no premium!)')
print('  Flash: $0.30/$2.50 per MTok input/output')


## Ex 3: google.genai SDK — Launch Tuning Job


In [ ]:
print('=== NEW google.genai SDK (use this!) ===')
print()
print('from google import genai')
print('from google.genai import types')
print()
print('client = genai.Client(')
print('    vertexai=True,')
print('    project="your-project-id",')
print('    location="us-central1"')
print(')')
print()
print('tuning_job = client.tunings.tune(')
print('    base_model="gemini-2.5-flash",')
print('    training_dataset=types.TuningDataset(')
print('        gcs_uri="gs://your-bucket/train.jsonl",')
print('    ),')
print('    config=types.CreateTuningJobConfig(')
print('        tuned_model_display_name="my-model",')
print('        epoch_count=4,')
print('        learning_rate_multiplier=1.0,')
print('        adapter_size=4,')
print('    ),')
print(')')


## Ex 4: Monitor + Use Tuned Model


In [ ]:
print('=== Monitor tuning job ===')
print()
print('import time')
print('while True:')
print('    job = client.tunings.get(name=tuning_job.name)')
print('    print(f"State: {job.state}")')
print('    if job.state == "JOB_STATE_SUCCEEDED": break')
print('    if job.state == "JOB_STATE_FAILED": raise Exception(job.error)')
print('    time.sleep(60)')
print()
print('=== Use tuned model ===')
print()
print('response = client.models.generate_content(')
print('    model=job.tuned_model.endpoint,')
print('    contents="Your prompt here"')
print(')')
print('print(response.text)')


## Ex 5: Hyperparameter Comparison


In [ ]:
configs = {
    'Default (Flash)': {'lr': 1.0, 'epochs': 'auto', 'adapter': 4},
    'Aggressive (Pro small)': {'lr': 10, 'epochs': 20, 'adapter': 4},
    'Conservative': {'lr': 0.5, 'epochs': 3, 'adapter': 2},
    'Max capacity': {'lr': 5, 'epochs': 10, 'adapter': 16},
}
print('Hyperparameter configurations:')
print()
for name, cfg in configs.items():
    print(f'  {name:25s}: LR={cfg["lr"]}, epochs={cfg["epochs"]}, adapter={cfg["adapter"]}')
print()
print('Google recommendations:')
print('  Pro + small dataset (<1K): LR=10, epochs=20')
print('  Pro + large dataset (>1K): LR=1-5, epochs=10')
print('  Flash + any dataset: defaults work well')
print('  Start with adapter_size=4, increase only if underfitting')


## Ex 6: DPO Preference Data Format


In [ ]:
print('DPO preference tuning data format:')
print()
dpo_example = {
    'contents': [
        {'role': 'user', 'parts': [{'text': 'Explain quantum computing simply'}]}
    ],
    'candidates': [
        {
            'content': {'role': 'model', 'parts': [{'text': 'Quantum computing uses qubits that can be 0, 1, or both at once...'}]},
            'preference': 'PREFERRED'
        },
        {
            'content': {'role': 'model', 'parts': [{'text': 'Quantum computing is a type of computation that harnesses quantum mechanical phenomena...'}]},
            'preference': 'NOT_PREFERRED'
        }
    ]
}
print(json.dumps(dpo_example, indent=2))
print()
print('DPO config: beta=0.1 (default), range 0.01-0.5')
print('Higher beta = more conservative shift toward preferred')


## Ex 7: Evaluation with Gen AI Eval Service


In [ ]:
print('=== Gen AI Evaluation Service ===')
print()
print('# Automated evaluation after tuning')
print('eval_result = client.evals.evaluate(')
print('    dataset=eval_dataset,')
print('    metrics=[')
print('        types.RubricMetric.GENERAL_QUALITY,')
print('        types.RubricMetric.INSTRUCTION_FOLLOWING,')
print('        types.Metric(name="bleu"),')
print('        types.Metric(name="rouge_l"),')
print('    ],')
print(')')
print()
print('Metric categories:')
print('  Computation: BLEU, ROUGE, exact_match')
print('  Rubric: GENERAL_QUALITY, TEXT_QUALITY, INSTRUCTION_FOLLOWING')
print('  Pairwise: base vs tuned direct comparison')
print('  Custom: PointwiseMetric with custom prompt templates')


## Ex 8: Gemini vs Gemma Decision Matrix


In [ ]:
print('Gemini vs Gemma: when to use which')
print()
for factor, gemini, gemma in [
    ('Tuning method','LoRA (managed)','LoRA/QLoRA/Full (self-managed)'),
    ('Weights','Proprietary (locked)','Open (downloadable)'),
    ('Deployment','Shared endpoint only','Anywhere (vLLM, on-prem)'),
    ('Setup time','~10 min','~2-4 hours'),
    ('Cost model','Per-token training','Per-GPU-hour'),
    ('SLA','None (not covered)','Self-managed'),
    ('Data sovereignty','Data crosses regions','Full control'),
    ('Best for','Quick experiments, API','Enterprise, on-prem, India'),
]: print(f'  {factor:18s}: Gemini={gemini:30s} | Gemma={gemma}')


---
## Recap
Vertex AI fine-tuning uses LoRA adapters under the hood — fast, cheap, but weights stay on Google's infrastructure. Five methods: SFT (primary), DPO (preferences), Continuous (iteration), Checkpoints (best state), Gemma (open weights). Mandatory SDK migration to google.genai by June 2026. Cost formula: tokens × epochs × rate, with NO inference premium (unlike OpenAI). Pro small datasets need aggressive hyperparameters (LR=10, epochs=20). Auto-deploy to shared endpoint only (no dedicated/private, no SLA). India: inference from asia-south1, fine-tuning likely in us-central1, DPDPA compliance via CMEK + VPC-SC + ZDR. For data sovereignty: Gemma + IndiaAI Mission GPUs.
